## Étape 1: Installation des bibliothèques d'analyse de sentiment

In [ ]:
# Installation des bibliothèques nécessaires
!pip install vaderSentiment textblob nltk pandas numpy matplotlib seaborn

## Étape 2: Imports et configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
import nltk

# Télécharger les ressources NLTK nécessaires
nltk.download('punkt')
nltk.download('brown')

# Configuration de l'affichage
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
sns.set_style('whitegrid')

## Étape 3: Chargement des données

In [ ]:
# Chargement des datasets
pr_comments_df = pd.read_parquet("hf://datasets/hao-li/AIDev/pr_comments.parquet")
pr_reviews_df = pd.read_parquet("hf://datasets/hao-li/AIDev/pr_reviews.parquet")
pr_review_comments_df = pd.read_parquet("hf://datasets/hao-li/AIDev/pr_review_comments_v2.parquet")

# Charger les PRs (pour identifier AI vs Human)
pull_request_df = pd.read_parquet("hf://datasets/hao-li/AIDev/pull_request.parquet")

print(f"Nombre de commentaires PR: {len(pr_comments_df)}")
print(f"Nombre de reviews: {len(pr_reviews_df)}")
print(f"Nombre de commentaires de review: {len(pr_review_comments_df)}")
print(f"Nombre de PRs: {len(pull_request_df)}")

## Étape 4: Fonctions d'analyse de sentiment

Nous utilisons deux approches complémentaires:

### 1. VADER (Valence Aware Dictionary and sEntiment Reasoner)
- Optimisé pour les textes courts et informels (comme les commentaires GitHub)
- Retourne 4 scores: négatif, neutre, positif, et composé
- Le score composé varie de -1 (très négatif) à +1 (très positif)

### 2. TextBlob
- Retourne la polarité (-1 à +1) et la subjectivité (0 à 1)
- Utile pour comparer avec VADER

In [ ]:
# Initialiser l'analyseur VADER
vader_analyzer = SentimentIntensityAnalyzer()

def analyze_sentiment_vader(text):
    """
    Analyse le sentiment avec VADER.
    Retourne un dictionnaire avec les scores: neg, neu, pos, compound
    """
    if pd.isna(text) or text == "":
        return {'neg': 0, 'neu': 0, 'pos': 0, 'compound': 0}
    
    scores = vader_analyzer.polarity_scores(str(text))
    return scores

def analyze_sentiment_textblob(text):
    """
    Analyse le sentiment avec TextBlob.
    Retourne un dictionnaire avec polarity et subjectivity
    """
    if pd.isna(text) or text == "":
        return {'polarity': 0, 'subjectivity': 0}
    
    blob = TextBlob(str(text))
    return {
        'polarity': blob.sentiment.polarity,
        'subjectivity': blob.sentiment.subjectivity
    }

def categorize_sentiment(compound_score):
    """
    Catégorise le sentiment selon le score composé VADER:
    - Positif: >= 0.05
    - Neutre: entre -0.05 et 0.05
    - Négatif: <= -0.05
    """
    if compound_score >= 0.05:
        return 'positif'
    elif compound_score <= -0.05:
        return 'négatif'
    else:
        return 'neutre'

print("✓ Fonctions d'analyse de sentiment définies")

## Étape 5: Test des fonctions avec des exemples

In [ ]:
# Exemples de commentaires pour tester
test_comments = [
    "Great job! This code is excellent and well-documented.",
    "This is terrible code. Please fix it immediately.",
    "The implementation looks correct. Added some minor suggestions.",
    "LGTM! Thanks for the contribution 👍",
    "This breaks the entire application. Did you even test this?"
]

print("="*80)
print("TESTS D'ANALYSE DE SENTIMENT")
print("="*80)

for i, comment in enumerate(test_comments, 1):
    vader_scores = analyze_sentiment_vader(comment)
    textblob_scores = analyze_sentiment_textblob(comment)
    category = categorize_sentiment(vader_scores['compound'])
    
    print(f"\n{i}. Commentaire: \"{comment}\"")
    print(f"   VADER Compound: {vader_scores['compound']:.3f} (neg={vader_scores['neg']:.2f}, neu={vader_scores['neu']:.2f}, pos={vader_scores['pos']:.2f})")
    print(f"   TextBlob Polarity: {textblob_scores['polarity']:.3f}, Subjectivity: {textblob_scores['subjectivity']:.3f}")
    print(f"   → Catégorie: {category.upper()}")

## Étape 6: Identifier les PRs IA vs Humaines

Le dataset contient un champ permettant d'identifier si une PR est générée par un agent IA.

In [ ]:
# Explorer les colonnes du dataset pull_request pour trouver l'indicateur AI
print("Colonnes dans pull_request_df:")
print(pull_request_df.columns.tolist())

print("\nAperçu des PRs:")
print(pull_request_df.head())

# Vérifier s'il existe une colonne pour identifier les PRs IA
ai_columns = [col for col in pull_request_df.columns if 'ai' in col.lower() or 'agent' in col.lower() or 'bot' in col.lower()]
print(f"\nColonnes potentielles pour identifier les PRs IA: {ai_columns}")

## Étape 7: Appliquer l'analyse de sentiment aux commentaires

Nous allons analyser tous les types de commentaires:
1. **pr_comments**: Commentaires généraux sur la PR
2. **pr_reviews**: Reviews complètes
3. **pr_review_comments**: Commentaires inline sur le code

In [ ]:
def add_sentiment_analysis(df, text_column='body'):
    """
    Ajoute les colonnes d'analyse de sentiment à un DataFrame.
    """
    print(f"Analyse de {len(df)} commentaires...")
    
    # Analyse VADER
    vader_results = df[text_column].apply(analyze_sentiment_vader)
    df['vader_neg'] = vader_results.apply(lambda x: x['neg'])
    df['vader_neu'] = vader_results.apply(lambda x: x['neu'])
    df['vader_pos'] = vader_results.apply(lambda x: x['pos'])
    df['vader_compound'] = vader_results.apply(lambda x: x['compound'])
    
    # Analyse TextBlob
    textblob_results = df[text_column].apply(analyze_sentiment_textblob)
    df['textblob_polarity'] = textblob_results.apply(lambda x: x['polarity'])
    df['textblob_subjectivity'] = textblob_results.apply(lambda x: x['subjectivity'])
    
    # Catégorisation
    df['sentiment_category'] = df['vader_compound'].apply(categorize_sentiment)
    
    print("✓ Analyse terminée")
    return df

# Note: L'exécution peut prendre quelques minutes selon la taille des datasets
print("Prêt à analyser les commentaires!")
print("\nPour lancer l'analyse, décommentez et exécutez:")
print("# pr_comments_with_sentiment = add_sentiment_analysis(pr_comments_df.copy())")
print("# pr_reviews_with_sentiment = add_sentiment_analysis(pr_reviews_df.copy())")
print("# pr_review_comments_with_sentiment = add_sentiment_analysis(pr_review_comments_df.copy())")

## Étape 8: Analyse d'un échantillon (pour test rapide)

In [ ]:
# Analyser un échantillon pour tester rapidement
sample_size = 1000

print(f"Analyse d'un échantillon de {sample_size} commentaires...\n")

# Échantillon de pr_comments
pr_comments_sample = pr_comments_df.sample(n=min(sample_size, len(pr_comments_df)), random_state=42).copy()
pr_comments_sample = add_sentiment_analysis(pr_comments_sample)

print("\n" + "="*80)
print("RÉSUMÉ DES SENTIMENTS (Échantillon)")
print("="*80)
print("\nDistribution des catégories de sentiment:")
print(pr_comments_sample['sentiment_category'].value_counts())
print(f"\nPourcentages:")
print(pr_comments_sample['sentiment_category'].value_counts(normalize=True) * 100)

print(f"\n\nStatistiques VADER Compound Score:")
print(pr_comments_sample['vader_compound'].describe())

print(f"\n\nStatistiques TextBlob Polarity:")
print(pr_comments_sample['textblob_polarity'].describe())

## Étape 9: Visualisation des résultats

In [ ]:
# Créer des visualisations
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Distribution des catégories de sentiment
pr_comments_sample['sentiment_category'].value_counts().plot(kind='bar', ax=axes[0, 0], color=['green', 'gray', 'red'])
axes[0, 0].set_title('Distribution des catégories de sentiment')
axes[0, 0].set_xlabel('Catégorie')
axes[0, 0].set_ylabel('Nombre de commentaires')
axes[0, 0].tick_params(axis='x', rotation=45)

# 2. Distribution du score VADER Compound
axes[0, 1].hist(pr_comments_sample['vader_compound'], bins=50, color='skyblue', edgecolor='black')
axes[0, 1].set_title('Distribution du VADER Compound Score')
axes[0, 1].set_xlabel('Compound Score')
axes[0, 1].set_ylabel('Fréquence')
axes[0, 1].axvline(x=0, color='red', linestyle='--', label='Neutre (0)')
axes[0, 1].legend()

# 3. Distribution de la polarité TextBlob
axes[1, 0].hist(pr_comments_sample['textblob_polarity'], bins=50, color='lightcoral', edgecolor='black')
axes[1, 0].set_title('Distribution de la polarité TextBlob')
axes[1, 0].set_xlabel('Polarité')
axes[1, 0].set_ylabel('Fréquence')
axes[1, 0].axvline(x=0, color='red', linestyle='--', label='Neutre (0)')
axes[1, 0].legend()

# 4. Relation entre polarité et subjectivité
axes[1, 1].scatter(pr_comments_sample['textblob_polarity'], 
                   pr_comments_sample['textblob_subjectivity'], 
                   alpha=0.5, s=10)
axes[1, 1].set_title('Polarité vs Subjectivité (TextBlob)')
axes[1, 1].set_xlabel('Polarité')
axes[1, 1].set_ylabel('Subjectivité')
axes[1, 1].axvline(x=0, color='red', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Visualisations générées")

## Étape 10: Exemples de commentaires par catégorie

In [ ]:
def show_examples_by_sentiment(df, n=3):
    """
    Affiche des exemples de commentaires pour chaque catégorie de sentiment.
    """
    for category in ['positif', 'neutre', 'négatif']:
        print(f"\n{'='*80}")
        print(f"EXEMPLES DE COMMENTAIRES {category.upper()}S")
        print(f"{'='*80}\n")
        
        examples = df[df['sentiment_category'] == category].nlargest(n, 'vader_compound' if category == 'positif' else 'vader_neg')
        
        for idx, row in examples.iterrows():
            print(f"Commentaire: {row['body'][:200]}..." if len(str(row['body'])) > 200 else f"Commentaire: {row['body']}")
            print(f"VADER Compound: {row['vader_compound']:.3f}")
            print(f"TextBlob Polarity: {row['textblob_polarity']:.3f}")
            print("-" * 80)

show_examples_by_sentiment(pr_comments_sample, n=3)

## Prochaines étapes:

1. **Lier les commentaires aux PRs (IA vs Humaine)**
   - Fusionner les DataFrames en utilisant les clés appropriées
   - Identifier la colonne qui distingue les PRs IA des PRs humaines

2. **Comparaison statistique**
   - Test t de Student pour comparer les moyennes
   - Test de Mann-Whitney U (non-paramétrique)
   - Analyse de l'effet size (Cohen's d)

3. **Contrôle des variables confondantes**
   - Complexité de la PR (nombre de fichiers modifiés, lignes de code)
   - Expérience du reviewer
   - Contexte du projet

4. **Visualisations comparatives**
   - Boxplots côte à côte
   - Distributions comparées
   - Graphiques de tendances

## Étape 11: Lier les commentaires aux PRs (IA vs Humaine)

Nous allons maintenant:
1. Identifier la colonne qui distingue les PRs IA des PRs humaines dans le dataset
2. Fusionner les commentaires avec les métadonnées des PRs
3. Créer un dataset unifié pour la comparaison statistique

In [ ]:
# Étape 11.1: Identifier le champ qui distingue les PRs IA des PRs humaines

# Explorer la structure de pull_request_df pour trouver le marqueur AI
print("="*80)
print("IDENTIFICATION DU MARQUEUR IA")
print("="*80)

# Vérifier la colonne 'agent' qui devrait indiquer le type de PR
if 'agent' in pull_request_df.columns:
    print("\n✓ Colonne 'agent' trouvée!")
    print("\nValeurs uniques dans 'agent':")
    print(pull_request_df['agent'].value_counts())
    
    # Identifier les agents IA vs humains
    ai_agents = pull_request_df[pull_request_df['agent'].notna()]['agent'].unique()
    print(f"\nAgents IA identifiés: {list(ai_agents)}")
    
    # Compter les PRs IA vs humaines
    ai_prs = pull_request_df[pull_request_df['agent'].notna()]
    human_prs = pull_request_df[pull_request_df['agent'].isna()]
    
    print(f"\nNombre de PRs IA: {len(ai_prs)}")
    print(f"Nombre de PRs humaines: {len(human_prs)}")
    print(f"Total PRs: {len(pull_request_df)}")
else:
    print("\n⚠ Colonne 'agent' non trouvée. Exploration des autres colonnes...")
    print(f"\nColonnes disponibles: {pull_request_df.columns.tolist()}")
    
# Créer une colonne binaire pour faciliter l'analyse
pull_request_df['is_ai_generated'] = pull_request_df['agent'].notna()
pull_request_df['pr_type'] = pull_request_df['is_ai_generated'].map({
    True: 'AI',
    False: 'Human'
})

print("\n" + "="*80)
print("Distribution des types de PRs:")
print(pull_request_df['pr_type'].value_counts())
print("="*80)

### Étape 11.2: Comprendre les clés de liaison

Avant de fusionner, identifions les colonnes de liaison entre les tables:
- **pr_comments** → **pull_request**: via `pr_id` ou `issue_id`
- **pr_reviews** → **pull_request**: via `pr_id`
- **pr_review_comments** → **pull_request**: via `pull_request_review_id` ou extraction depuis `pull_request_url`

In [ ]:
# Étape 11.2: Explorer les clés de liaison

print("="*80)
print("EXPLORATION DES CLÉS DE LIAISON")
print("="*80)

# Vérifier les colonnes dans chaque DataFrame
print("\n1. Colonnes dans pr_comments_df:")
print([col for col in pr_comments_df.columns if 'id' in col.lower() or 'pr' in col.lower()])
print(f"   Total colonnes: {pr_comments_df.columns.tolist()}")

print("\n2. Colonnes dans pr_reviews_df:")
print([col for col in pr_reviews_df.columns if 'id' in col.lower() or 'pr' in col.lower()])
print(f"   Total colonnes: {pr_reviews_df.columns.tolist()}")

print("\n3. Colonnes dans pr_review_comments_df:")
print([col for col in pr_review_comments_df.columns if 'id' in col.lower() or 'pr' in col.lower()])
print(f"   Total colonnes: {pr_review_comments_df.columns.tolist()}")

print("\n4. Colonnes dans pull_request_df:")
print([col for col in pull_request_df.columns if 'id' in col.lower()])
print(f"   Total colonnes: {pull_request_df.columns.tolist()}")

# Afficher quelques exemples pour comprendre la structure
print("\n" + "="*80)
print("EXEMPLES DE DONNÉES")
print("="*80)

print("\nExemple de pr_comments_df:")
print(pr_comments_df.head(2))

print("\nExemple de pr_reviews_df:")
print(pr_reviews_df.head(2))

print("\nExemple de pull_request_df:")
print(pull_request_df[['id', 'number', 'agent', 'pr_type']].head(2))

### Étape 11.3: Fusionner les commentaires avec les métadonnées des PRs

Maintenant que nous comprenons la structure, créons un dataset unifié qui combine:
- Les commentaires avec leur analyse de sentiment
- Les métadonnées des PRs (incluant si c'est IA ou humain)

In [ ]:
# Étape 11.3: Fusionner les commentaires avec les PRs

# Fonction pour fusionner et enrichir les commentaires
def merge_comments_with_pr_metadata(comments_df, pull_request_df, merge_key='pr_id'):
    """
    Fusionne un DataFrame de commentaires avec les métadonnées des PRs.
    
    Args:
        comments_df: DataFrame contenant les commentaires
        pull_request_df: DataFrame contenant les métadonnées des PRs
        merge_key: Nom de la colonne de liaison dans comments_df (par défaut 'pr_id')
    
    Returns:
        DataFrame fusionné avec les colonnes de PR enrichies
    """
    # Sélectionner les colonnes pertinentes de pull_request_df
    pr_metadata = pull_request_df[['id', 'agent', 'is_ai_generated', 'pr_type', 
                                     'title', 'number', 'repo_id', 'user']].copy()
    pr_metadata = pr_metadata.rename(columns={'id': 'pr_id', 'user': 'pr_author'})
    
    # Fusionner
    merged_df = comments_df.merge(
        pr_metadata,
        left_on=merge_key,
        right_on='pr_id',
        how='inner',
        suffixes=('_comment', '_pr')
    )
    
    return merged_df

print("="*80)
print("FUSION DES COMMENTAIRES AVEC LES MÉTADONNÉES DES PRs")
print("="*80)

# Note: On utilisera l'échantillon déjà analysé pour démonstration
# Pour l'analyse complète, appliquer sur les datasets complets

# Vérifier si l'échantillon existe
if 'pr_comments_sample' in locals() and 'vader_compound' in pr_comments_sample.columns:
    print("\n✓ Utilisation de l'échantillon déjà analysé")
    
    # Fusionner l'échantillon avec les métadonnées des PRs
    comments_with_pr_type = merge_comments_with_pr_metadata(
        pr_comments_sample, 
        pull_request_df
    )
    
    print(f"\nNombre de commentaires avant fusion: {len(pr_comments_sample)}")
    print(f"Nombre de commentaires après fusion: {len(comments_with_pr_type)}")
    print(f"Commentaires perdus (PRs non trouvées): {len(pr_comments_sample) - len(comments_with_pr_type)}")
    
    # Afficher la distribution
    print("\n" + "="*80)
    print("DISTRIBUTION DES COMMENTAIRES PAR TYPE DE PR")
    print("="*80)
    print(comments_with_pr_type['pr_type'].value_counts())
    
    # Afficher un aperçu
    print("\nAperçu du dataset fusionné:")
    print(comments_with_pr_type[['body', 'vader_compound', 'sentiment_category', 
                                   'pr_type', 'pr_author', 'title']].head(3))
    
else:
    print("\n⚠ L'échantillon n'a pas encore été analysé.")
    print("Veuillez d'abord exécuter l'Étape 8 pour créer 'pr_comments_sample'")
    
    # Créer quand même la structure pour les datasets complets
    print("\n📝 Fonction de fusion créée et prête à utiliser sur les datasets complets:")

### Étape 11.4: Statistiques descriptives par type de PR

Comparons maintenant les sentiments entre les commentaires sur les PRs IA vs PRs humaines.

In [ ]:
# Étape 11.4: Statistiques descriptives par type de PR

if 'comments_with_pr_type' in locals():
    print("="*80)
    print("COMPARAISON DES SENTIMENTS: PRs IA vs PRs HUMAINES")
    print("="*80)
    
    # Séparer les commentaires par type de PR
    ai_comments = comments_with_pr_type[comments_with_pr_type['pr_type'] == 'AI']
    human_comments = comments_with_pr_type[comments_with_pr_type['pr_type'] == 'Human']
    
    print(f"\nNombre de commentaires sur PRs IA: {len(ai_comments)}")
    print(f"Nombre de commentaires sur PRs humaines: {len(human_comments)}")
    
    # 1. Distribution des catégories de sentiment
    print("\n" + "="*80)
    print("1. DISTRIBUTION DES CATÉGORIES DE SENTIMENT")
    print("="*80)
    
    print("\n📊 PRs IA:")
    if len(ai_comments) > 0:
        ai_sentiment_dist = ai_comments['sentiment_category'].value_counts(normalize=True) * 100
        print(ai_sentiment_dist)
    else:
        print("Aucun commentaire trouvé")
    
    print("\n📊 PRs Humaines:")
    if len(human_comments) > 0:
        human_sentiment_dist = human_comments['sentiment_category'].value_counts(normalize=True) * 100
        print(human_sentiment_dist)
    else:
        print("Aucun commentaire trouvé")
    
    # 2. Statistiques du score VADER Compound
    print("\n" + "="*80)
    print("2. SCORES VADER COMPOUND (moyenne du sentiment)")
    print("="*80)
    
    if len(ai_comments) > 0 and len(human_comments) > 0:
        print("\nPRs IA:")
        print(f"  Moyenne: {ai_comments['vader_compound'].mean():.4f}")
        print(f"  Médiane: {ai_comments['vader_compound'].median():.4f}")
        print(f"  Écart-type: {ai_comments['vader_compound'].std():.4f}")
        
        print("\nPRs Humaines:")
        print(f"  Moyenne: {human_comments['vader_compound'].mean():.4f}")
        print(f"  Médiane: {human_comments['vader_compound'].median():.4f}")
        print(f"  Écart-type: {human_comments['vader_compound'].std():.4f}")
        
        # Différence
        diff_mean = ai_comments['vader_compound'].mean() - human_comments['vader_compound'].mean()
        print(f"\n📈 Différence de moyenne (IA - Humaine): {diff_mean:.4f}")
        
        if diff_mean > 0:
            print("   → Les commentaires sur les PRs IA sont légèrement plus POSITIFS")
        elif diff_mean < 0:
            print("   → Les commentaires sur les PRs IA sont légèrement plus NÉGATIFS")
        else:
            print("   → Pas de différence apparente")
    
    # 3. Statistiques du score TextBlob Polarity
    print("\n" + "="*80)
    print("3. SCORES TEXTBLOB POLARITY")
    print("="*80)
    
    if len(ai_comments) > 0 and len(human_comments) > 0:
        print("\nPRs IA:")
        print(f"  Moyenne: {ai_comments['textblob_polarity'].mean():.4f}")
        print(f"  Médiane: {ai_comments['textblob_polarity'].median():.4f}")
        
        print("\nPRs Humaines:")
        print(f"  Moyenne: {human_comments['textblob_polarity'].mean():.4f}")
        print(f"  Médiane: {human_comments['textblob_polarity'].median():.4f}")
        
        diff_polarity = ai_comments['textblob_polarity'].mean() - human_comments['textblob_polarity'].mean()
        print(f"\n📈 Différence de moyenne (IA - Humaine): {diff_polarity:.4f}")
    
    # 4. Récapitulatif
    print("\n" + "="*80)
    print("📋 RÉCAPITULATIF")
    print("="*80)
    print(f"\nDataset analysé: Échantillon de {len(comments_with_pr_type)} commentaires")
    print(f"PRs IA: {len(ai_comments)} commentaires ({len(ai_comments)/len(comments_with_pr_type)*100:.1f}%)")
    print(f"PRs Humaines: {len(human_comments)} commentaires ({len(human_comments)/len(comments_with_pr_type)*100:.1f}%)")
    
else:
    print("⚠ Veuillez d'abord exécuter l'étape 11.3 pour fusionner les données")

### Étape 11.5: Visualisations comparatives IA vs Humaine

Créons des graphiques pour visualiser les différences de sentiment entre les deux types de PRs.

In [ ]:
# Étape 11.5: Visualisations comparatives

if 'comments_with_pr_type' in locals():
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Séparer les données
    ai_comments = comments_with_pr_type[comments_with_pr_type['pr_type'] == 'AI']
    human_comments = comments_with_pr_type[comments_with_pr_type['pr_type'] == 'Human']
    
    # 1. Distribution des catégories de sentiment (barres côte à côte)
    if len(ai_comments) > 0 and len(human_comments) > 0:
        sentiment_comparison = pd.DataFrame({
            'AI': ai_comments['sentiment_category'].value_counts(normalize=True) * 100,
            'Human': human_comments['sentiment_category'].value_counts(normalize=True) * 100
        })
        sentiment_comparison.plot(kind='bar', ax=axes[0, 0], color=['#FF6B6B', '#4ECDC4'])
        axes[0, 0].set_title('Distribution des sentiments: IA vs Humaine (%)', fontsize=12, fontweight='bold')
        axes[0, 0].set_xlabel('Catégorie de sentiment')
        axes[0, 0].set_ylabel('Pourcentage (%)')
        axes[0, 0].legend(title='Type de PR')
        axes[0, 0].tick_params(axis='x', rotation=45)
    
    # 2. Boxplot des scores VADER Compound
    data_for_boxplot = pd.DataFrame({
        'Score': pd.concat([ai_comments['vader_compound'], human_comments['vader_compound']]),
        'Type': ['AI']*len(ai_comments) + ['Human']*len(human_comments)
    })
    
    if len(data_for_boxplot) > 0:
        import seaborn as sns
        sns.boxplot(data=data_for_boxplot, x='Type', y='Score', ax=axes[0, 1], 
                    palette={'AI': '#FF6B6B', 'Human': '#4ECDC4'})
        axes[0, 1].set_title('Distribution des scores VADER Compound', fontsize=12, fontweight='bold')
        axes[0, 1].set_ylabel('VADER Compound Score')
        axes[0, 1].axhline(y=0, color='red', linestyle='--', alpha=0.5, label='Neutre')
        axes[0, 1].legend()
    
    # 3. Histogrammes superposés des scores VADER
    if len(ai_comments) > 0:
        axes[1, 0].hist(ai_comments['vader_compound'], bins=30, alpha=0.6, 
                       label='AI', color='#FF6B6B', edgecolor='black')
    if len(human_comments) > 0:
        axes[1, 0].hist(human_comments['vader_compound'], bins=30, alpha=0.6, 
                       label='Human', color='#4ECDC4', edgecolor='black')
    axes[1, 0].set_title('Distributions des scores de sentiment', fontsize=12, fontweight='bold')
    axes[1, 0].set_xlabel('VADER Compound Score')
    axes[1, 0].set_ylabel('Fréquence')
    axes[1, 0].axvline(x=0, color='red', linestyle='--', alpha=0.5)
    axes[1, 0].legend()
    
    # 4. Violin plot pour voir la distribution détaillée
    if len(data_for_boxplot) > 0:
        sns.violinplot(data=data_for_boxplot, x='Type', y='Score', ax=axes[1, 1],
                      palette={'AI': '#FF6B6B', 'Human': '#4ECDC4'})
        axes[1, 1].set_title('Distribution détaillée (Violin plot)', fontsize=12, fontweight='bold')
        axes[1, 1].set_ylabel('VADER Compound Score')
        axes[1, 1].axhline(y=0, color='red', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()
    
    print("\n✓ Visualisations comparatives générées")
    
else:
    print("⚠ Veuillez d'abord exécuter l'étape 11.3 pour fusionner les données")

### Étape 11.6: Application sur les datasets complets

Pour une analyse complète, voici le code pour traiter tous les commentaires (pas seulement l'échantillon).

In [ ]:
# Étape 11.6: Pipeline complet pour les datasets complets

# ATTENTION: L'exécution peut prendre plusieurs minutes selon la taille des données

def process_complete_dataset(comments_df, pr_df, dataset_name="comments"):
    """
    Pipeline complet: analyse de sentiment + fusion avec métadonnées PR
    
    Args:
        comments_df: DataFrame de commentaires bruts
        pr_df: DataFrame pull_request avec métadonnées
        dataset_name: Nom du dataset pour le tracking
    
    Returns:
        DataFrame avec sentiment analysé et métadonnées PR
    """
    print(f"\n{'='*80}")
    print(f"TRAITEMENT DE: {dataset_name}")
    print(f"{'='*80}")
    
    # Étape 1: Analyse de sentiment
    print(f"\n[1/2] Analyse de sentiment sur {len(comments_df)} commentaires...")
    comments_analyzed = add_sentiment_analysis(comments_df.copy())
    
    # Étape 2: Fusion avec métadonnées PR
    print(f"\n[2/2] Fusion avec les métadonnées des PRs...")
    comments_enriched = merge_comments_with_pr_metadata(comments_analyzed, pr_df)
    
    print(f"\n✓ Traitement terminé!")
    print(f"  - Commentaires analysés: {len(comments_analyzed)}")
    print(f"  - Commentaires fusionnés: {len(comments_enriched)}")
    print(f"  - PRs IA: {len(comments_enriched[comments_enriched['pr_type']=='AI'])}")
    print(f"  - PRs Humaines: {len(comments_enriched[comments_enriched['pr_type']=='Human'])}")
    
    return comments_enriched

# Instructions pour traiter les datasets complets
print("="*80)
print("PIPELINE COMPLET - INSTRUCTIONS")
print("="*80)

print("""
Pour traiter les datasets complets, décommentez et exécutez:

# 1. Traiter tous les commentaires PR
all_pr_comments_enriched = process_complete_dataset(
    pr_comments_df, 
    pull_request_df, 
    "PR Comments"
)

# 2. Traiter toutes les reviews
all_pr_reviews_enriched = process_complete_dataset(
    pr_reviews_df, 
    pull_request_df, 
    "PR Reviews"
)

# 3. Traiter tous les commentaires de review
all_pr_review_comments_enriched = process_complete_dataset(
    pr_review_comments_df, 
    pull_request_df, 
    "PR Review Comments"
)

# 4. Combiner tous les types de commentaires en un seul dataset
all_comments_combined = pd.concat([
    all_pr_comments_enriched,
    all_pr_reviews_enriched,
    all_pr_review_comments_enriched
], ignore_index=True)

print(f"\\nTotal commentaires combinés: {len(all_comments_combined)}")

# 5. Sauvegarder les résultats
all_comments_combined.to_parquet('sentiment_analysis_results.parquet')
print("✓ Résultats sauvegardés dans 'sentiment_analysis_results.parquet'")
""")